# Assignment 05 — Optimization (100 points)

**Unit**: AI 200 — Mathematical Foundations for AI  
**Competition alignment**: USAAIO 2026 Round 1 & Round 2  

---

## Background

Gradient descent is the core optimization algorithm in machine learning. For a differentiable function $f$, the update rule $\mathbf{x}_{t+1} = \mathbf{x}_t - \eta \nabla f(\mathbf{x}_t)$ moves toward a local minimum. Momentum accelerates convergence on ill-conditioned problems by accumulating a velocity vector. The convergence rate for quadratics depends on the **condition number** $\kappa = \lambda_{\max}/\lambda_{\min}$ of the Hessian.

## Notation

| Symbol | Meaning |
|--------|---------|
| $\eta$ | learning rate |
| $\nabla f(\mathbf{x})$ | gradient at $\mathbf{x}$ |
| $L$ | Lipschitz constant of the gradient ($= \lambda_{\max}(\nabla^2 f)$ for quadratics) |
| $\kappa$ | condition number $\lambda_{\max}/\lambda_{\min}$ |
| $\beta$ | momentum coefficient |

In [ ]:
# DO NOT MAKE ANY CHANGE IN THIS CELL
import numpy as np
np.random.seed(42)
np.set_printoptions(precision=6, suppress=True)

> **WARNING**: Do not import any additional libraries. No `scipy.optimize` or PyTorch optimizers. All optimizers must be from scratch.

---

## Part 1 (25 points, coding task)

**Vanilla Gradient Descent**

Implement gradient descent with full trajectory tracking. Stop when $\|\nabla f\| < \text{tol}$.

Test on the quadratic $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^\top A \mathbf{x} + \mathbf{b}^\top \mathbf{x}$ where $A = \begin{bmatrix}4&1\\1&2\end{bmatrix}$ and $\mathbf{b} = [-3, -1]^\top$. The analytical minimum is $\mathbf{x}^* = -A^{-1}\mathbf{b}$.

*Reasoning is not required.*

In [ ]:
def gradient_descent(grad_f, x0: np.ndarray, lr: float,
                     max_iter: int = 1000, tol: float = 1e-8) -> dict:
    """Vanilla gradient descent.
    
    Args:
        grad_f: callable, takes shape (n,) -> shape (n,)
        x0: shape (n,) - starting point
        lr: learning rate
        max_iter: maximum iterations
        tol: stop when ||grad|| < tol
    
    Returns:
        dict with:
            'x_opt': shape (n,) - final point
            'trajectory': list of shape (n,) arrays
            'grad_norms': list of gradient norms
            'n_iter': int
    """
    ### WRITE YOUR SOLUTION HERE ###
    
    pass

""" END OF THIS PART """

In [ ]:
# Test
A = np.array([[4, 1], [1, 2]], dtype=float)    # (2, 2) — PD
b = np.array([-3, -1], dtype=float)             # (2,)

grad_f = lambda x: A @ x + b  # (2,)

result = gradient_descent(grad_f, x0=np.array([5.0, 5.0]), lr=0.2)

x_star = -np.linalg.solve(A, b)  # (2,)
print(f"GD solution:         {result['x_opt']}")
print(f"Analytical solution: {x_star}")
print(f"Converged in {result['n_iter']} iterations")
print(f"Final gradient norm: {result['grad_norms'][-1]:.2e}")

---

Momentum helps gradient descent navigate narrow valleys. Instead of stepping purely in the gradient direction, we maintain a velocity vector that accumulates past gradients:
$$\mathbf{v}_{t+1} = \beta \mathbf{v}_t + \nabla f(\mathbf{x}_t), \quad \mathbf{x}_{t+1} = \mathbf{x}_t - \eta \mathbf{v}_{t+1}$$

---

## Part 2 (25 points, coding task)

**Gradient Descent with Momentum**

Implement Polyak momentum. Compare convergence speed against vanilla GD on an ill-conditioned quadratic with condition number $\kappa = 100$ (eigenvalues 100 and 1).

*Reasoning is not required.*

In [ ]:
def gradient_descent_momentum(grad_f, x0: np.ndarray, lr: float,
                               beta: float = 0.9, max_iter: int = 1000,
                               tol: float = 1e-8) -> dict:
    """Gradient descent with Polyak momentum.
    
    Args:
        grad_f: gradient function, shape (n,) -> shape (n,)
        x0: shape (n,) - starting point
        lr: learning rate
        beta: momentum coefficient in [0, 1)
        max_iter, tol: stopping criteria
    
    Returns:
        Same dict format as gradient_descent
    """
    ### WRITE YOUR SOLUTION HERE ###
    
    pass

""" END OF THIS PART """

In [ ]:
# Ill-conditioned problem: kappa = 100
A_ill = np.array([[100, 0], [0, 1]], dtype=float)  # (2, 2)
b_ill = np.array([-10, -5], dtype=float)           # (2,)
grad_ill = lambda x: A_ill @ x + b_ill

result_gd = gradient_descent(grad_ill, np.array([5.0, 5.0]), lr=0.015)
result_mom = gradient_descent_momentum(grad_ill, np.array([5.0, 5.0]), lr=0.015, beta=0.9)

x_star_ill = -np.linalg.solve(A_ill, b_ill)  # (2,)
print(f"Analytical: {x_star_ill}")
print(f"GD:       {result_gd['x_opt']} in {result_gd['n_iter']} iters")
print(f"Momentum: {result_mom['x_opt']} in {result_mom['n_iter']} iters")

---

For a quadratic $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^\top A\mathbf{x} + \mathbf{b}^\top\mathbf{x}$, gradient descent converges if and only if $\eta < 2/\lambda_{\max}(A)$, and the optimal rate is $\eta^* = 2/(\lambda_{\max} + \lambda_{\min})$. The convergence rate per iteration is $\left(\frac{\kappa - 1}{\kappa + 1}\right)^2$ where $\kappa$ is the condition number.

---

## Part 3 (25 points, mixed task)

**Learning Rate Analysis**

**(a)** (15 points, coding) For $A = \begin{bmatrix}4&1\\1&2\end{bmatrix}$: compute $L = \lambda_{\max}(A)$, then run GD with learning rates $\eta \in \{0.01, 0.05, 0.1, 1/L, 0.3, 0.4, 2/L, 0.6\}$. For each, report: converged (Y/N), iterations, final error. Verify that $\eta > 2/L$ diverges. *Reasoning is not required.*

**(b)** (10 points, non-coding) Prove that for the quadratic $f(\mathbf{x}) = \frac{1}{2}\mathbf{x}^\top A\mathbf{x}$, gradient descent with step size $\eta$ satisfies $\|\mathbf{x}_{t+1} - \mathbf{x}^*\|_A \leq \max(|1 - \eta\lambda_{\min}|, |1 - \eta\lambda_{\max}|) \cdot \|\mathbf{x}_t - \mathbf{x}^*\|_A$. Use the eigendecomposition $A = Q\Lambda Q^\top$. *Reasoning is required.*

In [ ]:
# Part 3a
### WRITE YOUR SOLUTION HERE ###

pass

### WRITE YOUR SOLUTION HERE (Part 3b) ###



""" END OF THIS PART """

---

Gradient descent is the standard way to solve large-scale linear regression, where the closed-form solution $(X^\top X)^{-1}X^\top\mathbf{y}$ is too expensive to compute.

---

## Part 4 (25 points, coding task)

**Gradient Descent for Linear Regression**

Solve $\min_{\mathbf{w}} \frac{1}{2N}\|X\mathbf{w} - \mathbf{y}\|^2$ using gradient descent.

The gradient is $\nabla_{\mathbf{w}} L = \frac{1}{N}X^\top(X\mathbf{w} - \mathbf{y})$.

Choose an appropriate learning rate based on the Lipschitz constant $L = \lambda_{\max}(X^\top X)/N$, run from $\mathbf{w}_0 = \mathbf{0}$, and compare with the closed-form $\mathbf{w}^* = (X^\top X)^{-1}X^\top\mathbf{y}$.

*Reasoning is not required.*

In [ ]:
# Generate linear regression data
N, d = 100, 5
X = np.random.randn(N, d)                                    # (100, 5)
w_true = np.array([1, -2, 3, 0.5, -1], dtype=float)          # (5,)
y = X @ w_true + 0.5 * np.random.randn(N)                    # (100,)

### WRITE YOUR SOLUTION HERE ###
# 1. Define gradient function for MSE loss
# 2. Choose learning rate (hint: 1/L where L = max eigenvalue of X^T X / N)
# 3. Run gradient descent from w0 = zeros
# 4. Compute closed-form solution
# 5. Print: w_true, w_gd, w_closed_form, and errors

pass

""" END OF THIS PART """